In [0]:
from pyspark.sql.functions import *

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
df = spark.table('customer360_cata.bronze.web_activities')

In [0]:
df = df.withColumn(
    "session_time",
    coalesce(
        try_to_date(col("session_time"), "yyyy/MM/dd"),
        try_to_date(col("session_time"), "dd-MM-yyyy"),
        try_to_date(col("session_time"), "yyyy-MM-dd"),
        try_to_date(col("session_time"), "yyyyMMdd"),
        try_to_date(col("session_time"), "MM-dd-yyyy"),
        try_to_date(col("session_time"), "dd/MM/yyyy"),
    )
)

In [0]:
df = df.withColumn('page_viewed',lower(col('page_viewed')))\
.withColumn('device_type',initcap(col('device_type')))

In [0]:
df =df.dropDuplicates(["session_id","customer_id"])\
    .dropna(subset=["session_id","customer_id"])

In [0]:
df =df.drop("_rescued_data")

In [0]:
df = df.withColumn("customer_id", col("customer_id").cast("int"))

In [0]:
df.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true") \
    .option('path',base_adls2_path + '/silver/web_activities') \
        .saveAsTable('customer360_cata.silver.web_activities')